# Optional v2 — UTC-day curtailment planning model
This notebook explains and inspects the reproducible pipeline in `src/gridtoev/daily_curtailment.py` and `src/gridtoev/daily_model.py`. It does **not** change v1. The prediction is made at 00:00 UTC for total curtailment over that UTC day; every weather input is an archived GFS forecast for a target hour made 24 hours earlier. EirGrid curtailment appears only on the label side of the join.

## 1. Build the clean table
From the repository root, run `python scripts/build_daily_curtailment_data.py` once to download/cache the public forecasts. The builder requires all 24 forecast hours in all four regions and 48 valid half-hour EirGrid labels. The committed table lets you inspect and retrain without re-downloading.

In [ ]:
import pandas as pd
from gridtoev.daily_curtailment import DEFAULT_DAILY_DATASET
from gridtoev.daily_model import DEFAULT_REPORT, DailyCurtailmentService, train_daily_model

data = pd.read_csv(DEFAULT_DAILY_DATASET, parse_dates=['issue_timestamp_utc', 'forecast_max_available_at_utc'])
print(f'{len(data)} complete UTC days, {data.issue_timestamp_utc.min()} through {data.issue_timestamp_utc.max()}')
print('Curtailment-event rate:', round(data.curtailment_event.mean(), 3))
data.head()

## 2. Check the as-of boundary
The latest forecast value used for each day must be known before that day's 00:00 UTC issue. This checks that invariant in the assembled table; the builder also checks every hourly input before aggregation.

In [ ]:
assert (data.forecast_max_available_at_utc <= data.issue_timestamp_utc).all()
data[['issue_timestamp_utc', 'forecast_max_available_at_utc', 'curtailment_mwh']].head()

## 3. Train, select on 2025, then evaluate on untouched 2026 days
The training function fits an event classifier and compares two MWh estimators on 2025 only. It refits the selected method through 2025 and evaluates on 2026. The report records always-zero and monthly-median daily baselines. **Do not compare these MWh errors with v1's half-hour dispatch-down MAE; the targets differ.**

In [ ]:
report = train_daily_model()
print('Selected amount method:', report['selected_amount_method'])
print('2026 model daily MAE:', round(report['test']['daily_mae_mwh'], 1), 'MWh')
print('2026 always-zero daily MAE:', round(report['test_zero_amount_baseline']['daily_mae_mwh'], 1), 'MWh')
print('2026 event AP:', round(report['test']['event_average_precision'], 3))
pd.DataFrame.from_dict(report['test_by_quarter'], orient='index')

## 4. Make an example prediction
This example uses an archived forecast row, so it is a retrospective demonstration, not a fresh live forecast. For the current UTC day, use `DailyCurtailmentService().predict_date(...)` or the opt-in API endpoint described in `docs/DAILY_CURTAILMENT_V2.md`.

In [ ]:
service = DailyCurtailmentService()
service.predict_features(data.iloc[[-1]])